In [10]:
import random
import pandas as pd
import numpy as np
import os
import IPython.display as ipd
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBRegressor

import warnings
warnings.filterwarnings(action='ignore') 

In [11]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

In [12]:
# 1. 강수량, 일조, 일사는 0으로 채우기 (비 안 옴, 해 안 뜸)
train['강수량(mm)'] = train['강수량(mm)'].fillna(0)
train['일조(hr)'] = train['일조(hr)'].fillna(0)
train['일사(MJ/m2)'] = train['일사(MJ/m2)'].fillna(0)

# 기온, 풍속, 습도는 선형 보간으로 채움 (오류 수정됨)
cols_to_interp = ['기온(C)', '풍속(m/s)', '습도(%)']
train[cols_to_interp] = train[cols_to_interp].interpolate(method='linear')

# 결과 확인
print(train.isnull().sum())

num_date_time    0
건물번호             0
일시               0
기온(C)            0
강수량(mm)          0
풍속(m/s)          0
습도(%)            0
일조(hr)           0
일사(MJ/m2)        0
전력소비량(kWh)       0
dtype: int64


In [13]:
for data in [train, test]:
    data['일시'] = pd.to_datetime(data['일시'])
    data['year'] = data['일시'].dt.year
    data['month'] = data['일시'].dt.month
    data['day'] = data['일시'].dt.day
    data['hour'] = data['일시'].dt.hour
    data['dayofweek'] = data['일시'].dt.dayofweek

In [14]:
# 6월 6일 평균 대비 6월 7일 평균 전력의 비율 계산
june_6 = train[train['일시'].dt.strftime('%m-%d') == '06-06'].groupby('건물번호')['전력소비량(kWh)'].mean()
june_7 = train[train['일시'].dt.strftime('%m-%d') == '06-07'].groupby('건물번호')['전력소비량(kWh)'].mean()

drop_ratio = (june_6 / june_7)
# 비율이 0.7 미만(30% 이상 급감)인 건물들만 보기
sensitive_buildings = drop_ratio[drop_ratio < 0.7].index.tolist()

print(f"공휴일에 민감한 건물 번호: {sensitive_buildings}")

공휴일에 민감한 건물 번호: [3, 17, 19, 20, 53, 54, 59, 60, 74, 77, 78, 80, 82, 83, 84]


In [15]:
# train['is_holiday'] = train['일시'].dt.strftime('%m-%d').isin(['06-06'])
train['is_holiday'] = train['일시'].dt.strftime('%m-%d').isin(['06-06', '08-15']).astype(int)
train['is_holiday'] = ((train['is_holiday'] == 1) | (train['dayofweek'] >= 5)).astype(int)

In [16]:
train['is_weekend'] = train['일시'].dt.dayofweek >= 5

In [17]:
# 불쾌지수 계산 함수 정의
def calculate_di(temp, humid):
    return 1.8 * temp - 0.55 * (1 - humid/100) * (1.8 * temp - 26) + 32

# 피처 생성
train['DI'] = calculate_di(train['기온(C)'], train['습도(%)'])
test['DI'] = calculate_di(test['기온(C)'], test['습도(%)'])

# 확인용
print(train[['기온(C)', '습도(%)', 'DI']].head())

   기온(C)  습도(%)        DI
0   18.6   42.0  63.09388
1   18.0   45.0  62.46400
2   17.7   45.0  62.08735
3   16.7   48.0  60.89884
4   18.4   43.0  62.88788


In [18]:
# [1] 공통 피처 리스트 정의 (순서가 중요합니다!)
features = [
    '건물번호', '기온(C)', '강수량(mm)', '풍속(m/s)', '습도(%)', 'DI', 
    'month', 'day', 'hour', 'dayofweek', 'is_holiday'
]

# [2] Test 데이터 전처리 (Train은 이미 되어 있다고 가정)
test['일시'] = pd.to_datetime(test['일시'])
test['month'] = test['일시'].dt.month
test['day'] = test['일시'].dt.day
test['hour'] = test['일시'].dt.hour
test['dayofweek'] = test['일시'].dt.weekday

# 공휴일 생성 (현충일, 광복절 + 주말)
test['is_holiday'] = test['일시'].dt.strftime('%m-%d').isin(['06-06', '08-15']).astype(int)
test['is_holiday'] = ((test['is_holiday'] == 1) | (test['dayofweek'] >= 5)).astype(int)

# DI(불쾌지수) 생성
test['DI'] = 1.8 * test['기온(C)'] - 0.55 * (1 - test['습도(%)']/100) * (1.8 * test['기온(C)'] - 26) + 32

from sklearn.model_selection import TimeSeriesSplit

# 1. SMAPE 평가 지표 함수 정의
def smape(y_true, y_pred):
    return 100 * np.mean(2 * np.abs(y_true - y_pred) / (np.abs(y_true) + np.abs(y_pred)))

# 2. TimeSeriesSplit을 위해 데이터 정렬 (핵심!)
# 시계열 순서가 섞이지 않도록 일시 및 건물번호 기준으로 정렬합니다.
train = train.sort_values(['일시', '건물번호']).reset_index(drop=True)

X_train = train[features]
y_train = train['전력소비량(kWh)']
X_test = test[features]

print(f"✅ 데이터 준비 완료! (Train: {X_train.shape}, Test: {X_test.shape})")

# 3. TimeSeriesSplit 설정
n_splits = 5
tscv = TimeSeriesSplit(n_splits=n_splits)

# 폴드별 예측값을 누적할 배열 (OOF 앙상블)
test_preds = np.zeros(len(X_test))
val_scores = []

print("🚀 TimeSeriesSplit 교차 검증 시작...")

for fold, (train_idx, val_idx) in enumerate(tscv.split(X_train)):
    # 훈련용/검증용 데이터 분할
    X_tr, y_tr = X_train.iloc[train_idx], y_train.iloc[train_idx]
    X_val, y_val = X_train.iloc[val_idx], y_train.iloc[val_idx]
    
    # 모델 정의
    model = XGBRegressor(
        n_estimators=1000,
        learning_rate=0.05,
        max_depth=7,
        random_state=42,
        tree_method='hist',
        n_jobs=-1
    )
    
    # 조기 종료(Early Stopping)를 적용하여 과적합 방지 및 속도 향상
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        verbose=False # 로그가 너무 길어지는 것 방지
    )
    
    # 검증 세트 예측 및 SMAPE 계산
    val_preds = model.predict(X_val)
    fold_smape = smape(y_val, val_preds)
    val_scores.append(fold_smape)
    
    print(f"Fold {fold+1} SMAPE: {fold_smape:.4f}")
    
    # 테스트 세트 예측 (각 폴드의 예측값을 평균내어 최종 예측 앙상블)
    test_preds += model.predict(X_test) / n_splits

print(f"\n✨ 교차 검증 완료! 평균 Val SMAPE: {np.mean(val_scores):.4f}")

# [6] 제출용 파일 생성
submission = pd.read_csv('sample_submission.csv') 
submission['answer'] = test_preds

# 파일 저장
submission.to_csv('team_base_n.csv)', index=False)
print("🏁 제출 파일(team_base_n.csv)이 생성되었습니다!")

✅ 데이터 준비 완료! (Train: (204000, 11), Test: (16800, 11))
🚀 TimeSeriesSplit 교차 검증 시작...
Fold 1 SMAPE: 16.3903
Fold 2 SMAPE: 18.9475
Fold 3 SMAPE: 9.1435
Fold 4 SMAPE: 9.7921
Fold 5 SMAPE: 8.5687

✨ 교차 검증 완료! 평균 Val SMAPE: 12.5684
🏁 제출 파일(team_base_n.csv)이 생성되었습니다!
